# FarmSense — Fine-tuning Gemma 4 avec Unsloth
## Gemma 4 Good Hackathon | Kaggle x Google DeepMind

Ce notebook fine-tune Gemma 4 e4b sur notre dataset agricole Sénégal/Wolof.
On utilise Unsloth QLoRA pour réduire la mémoire GPU et accélérer l'entraînement.

Résultat : un modèle FarmSense-Gemma4 spécialisé qui connaît vraiment
les maladies du Sahel, le Wolof agricole, et donne des réponses structurées.

Prérequis : GPU T4 x2 activé dans les paramètres Kaggle.


In [ ]:
# ── 1. Installer Unsloth et les dépendances ──────────────────────────
import os

os.system('pip install -q unsloth')
os.system('pip install -q transformers datasets peft trl accelerate bitsandbytes')
print('OK Dépendances installées')


In [ ]:
# ── 2. Cloner le repo FarmSense et générer le dataset ────────────────
GITHUB_URL = 'https://github.com/TON_USERNAME/farmsense'  # <- à modifier

if not os.path.exists('/kaggle/working/farmsense'):
    os.system(f'git clone {GITHUB_URL} /kaggle/working/farmsense')
else:
    os.system('git -C /kaggle/working/farmsense pull')

# Générer le dataset JSONL
os.chdir('/kaggle/working/farmsense/training')
os.system('python3 generate_dataset.py')
print('OK Dataset prêt')


In [ ]:
# ── 3. Charger le modèle avec Unsloth ───────────────────────────────
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'unsloth/gemma-4-8b-it-unsloth-bnb-4bit'  # version 4bit quantisée
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,       # QLoRA 4bit — économise la mémoire GPU
    dtype=None,              # auto-détection
)
print(f'OK Modèle chargé : {MODEL_NAME}')


In [ ]:
# ── 4. Configurer LoRA ───────────────────────────────────────────────
# LoRA ajoute des petites matrices d'adaptation sans toucher au modèle de base
# r=16 est un bon compromis qualité/vitesse pour notre dataset

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                        # rang LoRA
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('OK LoRA configuré')
print(f'Paramètres entraînables : {model.num_parameters(only_trainable=True):,}')


In [ ]:
# ── 5. Charger et préparer le dataset ───────────────────────────────
import json
from datasets import Dataset

# Lire le JSONL
examples = []
with open('/kaggle/working/farmsense/training/farmsense_dataset.jsonl', 'r') as f:
    for line in f:
        examples.append(json.loads(line))

print(f'Exemples chargés : {len(examples)}')

# Convertir en format chat template Gemma 4
def format_example(example):
    text = tokenizer.apply_chat_template(
        example['conversations'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

dataset = Dataset.from_list(examples)
dataset = dataset.map(format_example, remove_columns=['conversations'])

print(f'Dataset formaté : {len(dataset)} exemples')
print('Exemple :')
print(dataset[0]['text'][:300])


In [ ]:
# ── 6. Entraînement avec SFTTrainer (Unsloth) ───────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,           # 3 epochs sur notre petit dataset
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        output_dir='/kaggle/working/farmsense-gemma4',
        report_to='none',
    ),
)

print('Démarrage du fine-tuning...')
trainer_stats = trainer.train()
print(f'Fine-tuning terminé !')
print(f'Durée : {trainer_stats.metrics["train_runtime"]:.0f} secondes')


In [ ]:
# ── 7. Tester le modèle fine-tuné ────────────────────────────────────
FastLanguageModel.for_inference(model)  # mode inférence rapide

test_messages = [
    {"role": "system", "content": "Tu es FarmSense, un assistant agricole pour les agriculteurs du Sénégal. Tu es direct, pratique, sans markdown, maximum 8 lignes."},
    {"role": "user",   "content": "[Zone: Kaolack | Langue: Français]\nMes feuilles de mil jaunissent depuis le bas avec une poudre grise."}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt'
).to('cuda')

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=300,
    temperature=0.2,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print('=== Réponse FarmSense fine-tuné ===')
print(response)


In [ ]:
# ── 8. Sauvegarder le modèle en GGUF pour Ollama ─────────────────────
# On exporte en GGUF q4_k_m pour une inférence rapide avec Ollama

SAVE_PATH = '/kaggle/working/farmsense-gemma4-gguf'

print('Export en GGUF q4_k_m...')
model.save_pretrained_gguf(
    SAVE_PATH,
    tokenizer,
    quantization_method='q4_k_m'  # bon compromis qualité/taille
)

import os
files = os.listdir(SAVE_PATH)
print(f'Fichiers exportés : {files}')
print('OK Modèle GGUF prêt pour Ollama')


In [ ]:
# ── 9. Créer un Modelfile Ollama et charger le modèle ────────────────
import subprocess, os

GGUF_FILE = os.path.join(SAVE_PATH, 'unsloth.Q4_K_M.gguf')

# Créer le Modelfile Ollama
modelfile_content = f'''FROM {GGUF_FILE}

SYSTEM "Tu es FarmSense, un assistant agricole pour les petits agriculteurs du Sénégal et du Sahel. Tu parles Français et Wolof. Tu es direct et pratique. RÈGLES : Jamais de markdown. Texte simple. Maximum 8 lignes. Structure : diagnostic / cause / actions numérotées / action immédiate."

PARAMETER temperature 0.2
PARAMETER num_ctx 4096
'''

with open('/kaggle/working/Modelfile', 'w') as f:
    f.write(modelfile_content)

# Installer Ollama si absent
if not os.path.exists('/usr/local/bin/ollama'):
    os.system('apt-get install -y zstd > /dev/null 2>&1')
    os.system('curl -fsSL https://ollama.com/install.sh | sh')

# Démarrer Ollama
import time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Créer le modèle Ollama
os.system('ollama create farmsense-gemma4 -f /kaggle/working/Modelfile')
print('OK Modèle farmsense-gemma4 créé dans Ollama')


In [ ]:
# ── 10. Tester via Ollama et lancer FarmSense avec le nouveau modèle ─
import requests, os, sys

# Test rapide
r = requests.post(
    'http://localhost:11434/api/chat',
    json={
        'model': 'farmsense-gemma4',
        'messages': [{
            'role': 'user',
            'content': '[Zone: Kaolack | Langue: Français]\nMes feuilles de mil jaunissent depuis le bas.'
        }],
        'stream': False
    },
    timeout=60
)
print('=== Test modèle fine-tuné ===')
print(r.json()['message']['content'])

# Lancer FarmSense avec le modèle fine-tuné
os.environ['GEMMA_MODEL'] = 'farmsense-gemma4'
sys.path.insert(0, '/kaggle/working/farmsense/app')
os.chdir('/kaggle/working/farmsense/app')

from pyngrok import ngrok, conf
NGROK_TOKEN = 'TON_TOKEN_NGROK'  # <- à modifier
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(5000)
print(f'\nFarmSense fine-tuné accessible : {public_url}')

import subprocess
subprocess.Popen(['python', 'app_flask.py'], cwd='/kaggle/working/farmsense/app')
print('FarmSense lancé avec le modèle fine-tuné !')
